# Earnings Call Sentiment — EDA & Model Training

End-to-end notebook: data exploration, model fine-tuning, evaluation, and error analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

plt.style.use('dark_background')
pd.set_option('display.max_colwidth', 120)

## 1. Load & Inspect Dataset

In [ ]:
df = pd.read_csv('../data/processed/transcripts.csv')
print(f'Total samples: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_counts = df['label'].value_counts()
axes[0].bar(label_counts.index, label_counts.values,
            color=['#ef4444', '#f59e0b', '#22d3a0'])
axes[0].set_title('Label Distribution')
axes[0].set_ylabel('Count')

df['text_length'] = df['text'].str.len()
for label, color in zip(['negative','neutral','positive'], ['#ef4444','#f59e0b','#22d3a0']):
    subset = df[df['label'] == label]['text_length']
    axes[1].hist(subset, bins=40, alpha=0.6, label=label, color=color)
axes[1].set_title('Transcript Length by Class')
axes[1].set_xlabel('Characters')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eval_output/eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Fine-Tune Longformer

In [ ]:
import torch
from transformers import LongformerTokenizer, LongformerForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

LABEL_MAP = {'negative': 0, 'neutral': 1, 'positive': 2}
MODEL_NAME = 'allenai/longformer-base-4096'
MAX_LEN = 1024
BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
class EarningsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.enc = tokenizer(texts, truncation=True, padding=True,
                              max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['labels'] = self.labels[idx]
        return item

tokenizer = LongformerTokenizer.from_pretrained(MODEL_NAME)
texts = df['text'].tolist()
labels = df['label'].map(LABEL_MAP).tolist()

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)

train_ds = EarningsDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = EarningsDataset(X_val,   y_val,   tokenizer, MAX_LEN)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
print(f'Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')

In [ ]:
model = LongformerForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_dl) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps)

train_losses, val_f1s = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        outputs.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += outputs.loss.item()

    avg_loss = total_loss / len(train_dl)
    train_losses.append(avg_loss)

    from sklearn.metrics import f1_score
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            preds = model(**batch).logits.argmax(-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(batch['labels'].cpu().tolist())

    val_f1 = f1_score(all_labels, all_preds, average='weighted')
    val_f1s.append(val_f1)
    print(f'Epoch {epoch}/{EPOCHS} | loss={avg_loss:.4f} | val_f1={val_f1:.4f}')

model.save_pretrained('../models/fine_tuned')
tokenizer.save_pretrained('../models/fine_tuned')
print('Model saved.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, EPOCHS+1), train_losses, 'o-', color='#4f8ef7')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(range(1, EPOCHS+1), val_f1s, 'o-', color='#22d3a0')
axes[1].set_title('Validation Weighted F1')
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('../data/eval_output/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Evaluation & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(all_labels, all_preds,
      target_names=['negative','neutral','positive']))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['negative','neutral','positive'],
            yticklabels=['negative','neutral','positive'])
plt.title('Confusion Matrix — Longformer Fine-Tuned')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('../data/eval_output/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Event-Study Backtest

In [ ]:
import yfinance as yf
from datetime import datetime, timedelta

EVENTS = [
    {'ticker': 'AAPL', 'date': '2024-01-25', 'sentiment': 'positive'},
    {'ticker': 'MSFT', 'date': '2024-01-30', 'sentiment': 'positive'},
    {'ticker': 'INTC', 'date': '2024-01-25', 'sentiment': 'negative'},
    {'ticker': 'META', 'date': '2024-02-01', 'sentiment': 'positive'},
    {'ticker': 'SNAP', 'date': '2024-02-06', 'sentiment': 'negative'},
]

results = []
for ev in EVENTS:
    dt = datetime.strptime(ev['date'], '%Y-%m-%d')
    start = (dt - timedelta(days=3)).strftime('%Y-%m-%d')
    end   = (dt + timedelta(days=8)).strftime('%Y-%m-%d')
    df_p = yf.download(ev['ticker'], start=start, end=end,
                       auto_adjust=True, progress=False)
    if df_p.empty: continue
    closes = df_p['Close'].dropna()
    idx = closes.index.searchsorted(pd.Timestamp(dt))
    if idx + 5 >= len(closes): continue
    base = float(closes.iloc[idx])
    r1d  = (float(closes.iloc[idx+1]) / base - 1) * 100
    r5d  = (float(closes.iloc[idx+5]) / base - 1) * 100
    results.append({**ev, 'r1d': round(r1d, 2), 'r5d': round(r5d, 2)})

bt_df = pd.DataFrame(results)
print(bt_df.to_string(index=False))

In [ ]:
bt_df['correct'] = (
    (bt_df['sentiment'] == 'positive') & (bt_df['r1d'] > 0) |
    (bt_df['sentiment'] == 'negative') & (bt_df['r1d'] < 0)
)
win_rate = bt_df['correct'].mean() * 100
print(f'Win Rate: {win_rate:.1f}%')

avg_by_class = bt_df.groupby('sentiment')['r1d'].mean()
print('\nAvg 1D Return by Sentiment:')
print(avg_by_class)

avg_by_class.plot(kind='bar', color=['#22d3a0','#f59e0b','#ef4444'],
                  figsize=(7, 4), title='Avg 1D Return by Sentiment Class')
plt.ylabel('Return (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../data/eval_output/backtest_returns.png', dpi=150, bbox_inches='tight')
plt.show()